# 📰 Fake News Detection using Machine Learning & NLP
### **Course:** Mini-Project / Machine Learning (2nd Year B.Tech CSE)
### **Objective:** Build a robust Natural Language Processing classifier to detect fabricated/fake news articles using TF-IDF and Classifiers (Passive Aggressive, Logistic Regression, Naive Bayes, Random Forest).

---
## Pipeline Overview:
1. **Exploratory Data Analysis (EDA)**
2. **Text Preprocessing (Stopwords, Regex, Porter Stemmer)**
3. **Feature Extraction (TF-IDF Vectorization - Unigrams & Bigrams)**
4. **Multi-Model Training & 5-Fold Cross Validation**
5. **Evaluation Metrics (Accuracy, Precision, Recall, F1, Confusion Matrix)**
6. **Model Serialization & Inference**

In [ ]:
# 1. Import Essential Libraries
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

nltk.download('stopwords', quiet=True)
print(" Libraries successfully imported!")

In [ ]:
# 2. Load Dataset
df = pd.read_csv('data/sample_news_dataset.csv')
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# 3. Check Class Balance
print(df['label'].value_counts())
sns.countplot(data=df, x='label', palette=['#eb3349', '#0ba360'])
plt.title("Class Distribution: Fake (0) vs Real (1)")
plt.show()

In [ ]:
# 4. Text Cleaning and Preprocessing Function
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_news_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs, HTML, and Special characters
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    
    words = text.split()
    stemmed = [ps.stem(w) for w in words if w not in stop_words and len(w) > 2]
    return " ".join(stemmed)

# Combine Title + Body
df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['cleaned_text'] = df['combined'].apply(clean_news_text)
df[['title', 'cleaned_text', 'label']].head()

In [ ]:
# 5. Train-Test Split & TF-IDF Vectorization
X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(f"X_train shape: {X_train_vec.shape}")
print(f"X_test shape: {X_test_vec.shape}")

In [ ]:
# 6. Multi-Model Benchmark
classifiers = {
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000, C=0.5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.5),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
}

results = []
for name, model in classifiers.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results.append({"Model": name, "Accuracy (%)": round(acc*100, 2), "F1-Score (%)": round(f1*100, 2)})

results_df = pd.DataFrame(results)
results_df

In [ ]:
# 7. Best Model Confusion Matrix Visualization
best_clf = PassiveAggressiveClassifier(max_iter=1000, C=0.5, random_state=42)
best_clf.fit(X_train_vec, y_train)
y_pred = best_clf.predict(X_test_vec)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Fake (0)', 'Predicted Real (1)'],
            yticklabels=['Actual Fake (0)', 'Actual Real (1)'])
plt.title("Passive Aggressive Classifier - Confusion Matrix")
plt.show()

print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 8. Live Prediction Test Function
def predict_news_authenticity(news_headline, news_content):
    raw = f"{news_headline} {news_content}"
    clean = clean_news_text(raw)
    vec = tfidf.transform([clean])
    pred = best_clf.predict(vec)[0]
    
    status = " REAL / AUTHENTIC NEWS" if pred == 1 else "🚨 FAKE / MISINFORMATION"
    print(f"\nInput Headline: {news_headline}")
    print(f"Prediction: {status}")

# Test on samples
predict_news_authenticity(
    "NASA Telescope Discovers Water Vapor on Distant Exoplanet",
    "Astronomers confirmed the presence of atmospheric water vapor using spectroscopy data."
)

predict_news_authenticity(
    "SHOCKING: Secret Government Experiment Spraying Microchips in Water Supply!",
    "Whistleblower exposes classified secret plot to control citizens through tap water."
)